# 🧍 Dense Body-Surface Feature Extraction with SAM 3

**Project context.** This notebook is the *successor* of the YOLOv8-Pose pipeline used in
Cif & Vasques (2026), *Video-based phenotyping using markerless pose estimation and a tabular
foundation model: inference from adult to pediatric hyperkinetic movement disorders*.

The original pipeline turned each clinical video into a sparse 17-keypoint skeleton
(YOLOv8x-pose-p6, COCO format). For this new project, we replace YOLOv8 by **Meta SAM 3
(November 2025)** to obtain a much richer description of the patient's body:

- a **dense segmentation mask** of the patient over time,
- **N points along the body silhouette (outer surface)**,
- a **K×L grid of points sampled inside the body (inner surface)**,
- classical geometric descriptors (centroid, area, perimeter, bounding box, principal axes).

All these signals are exported into a single Excel file per video, with one row per frame —
exactly the same downstream contract as the YOLO notebook, so that the file can be plugged
into the existing **TabICLv2** prediction pipeline without further changes.

**Three output videos are produced per input video** to support different inspection needs:

| Suffix              | Content                                                          |
|---------------------|------------------------------------------------------------------|
| `_overlay.mp4`      | original frame + coloured patient mask + contour & grid points  |
| `_mask.mp4`         | binary patient mask only (silhouette on black background)       |
| `_sidebyside.mp4`   | original \| mask \| overlay, the three side-by-side             |

**Optional SAM 3D Body section.** A separate (optional) section at the end of the notebook
uses **Meta SAM 3D Body** (released the same day as SAM 3, 19 November 2025) to reconstruct
a *full 3D mesh* of the patient on selected keyframes (default: 1 frame per second). The
patient mask produced by SAM 3 is fed to SAM 3D Body as a prompt, ensuring the 3D
reconstruction is locked onto the patient and not the neurologist.

**Built-in matplotlib visualisations.** The notebook includes:

- a 4-panel quicklook on a single frame (raw / mask / point cloud / final overlay),
- a 6-panel temporal feature summary (centroid trajectory, area, orientation, perimeter,
  aspect ratio, solidity) plus a per-grid-point motion-variance heatmap,
- a **3D space-time view** of the surface point cloud (axis Z = time), useful to spot
  oscillatory phenomenologies like tremor or chorea visually,
- (in the optional SAM 3D Body section) a 3D mesh viewer with a rotating animation.

## Dataset folder layout

The notebook expects a participant-organised dataset:

```
dataset/
├── P1/                     <- patient 1: 'P' + integer id
│   ├── rest.mp4
│   ├── posture.mp4
│   └── action.mp4
├── P2/
│   └── exam.mp4
├── ...
├── C1/                     <- healthy control 1: 'C' + integer id
│   └── exam.mp4
├── C2/
│   └── exam.mp4
└── manual_targets.csv      <- OPTIONAL, see 'Manual intervention' below
```

Each subject folder name is automatically parsed as `(subject_type, subject_id)` where
`subject_type ∈ {'patient', 'control'}` and `subject_id` is the integer that follows.
Both fields are propagated as columns into every Excel time series so that downstream
patient-grouped cross-validation in TabICLv2 (as in Cif & Vasques 2026) is trivial.

Outputs mirror the input layout:

```
outputs/
├── P1/
│   ├── rest_sam3_overlay.mp4
│   ├── rest_sam3_mask.mp4
│   ├── rest_sam3_sidebyside.mp4
│   ├── rest_sam3_timeseries.xlsx
│   └── ...
├── P2/...
├── C1/...
├── _summary.csv            <- per-video automation sanity report
└── _all_timeseries.xlsx    <- concatenated table ready for TabICLv2
```

## Manual intervention?

**No** — the pipeline is fully automatic end-to-end:

- automatic discovery of all `.mp4` under `dataset/`,
- automatic patient selection on frame 0 (largest 'person' mask),
- automatic patient re-identification on subsequent frames (IoU with previous frame),
- automatic per-video output generation,
- automatic generation of `_summary.csv` and concatenated `_all_timeseries.xlsx`.

**Optional safety net.** If on a few videos the neurologist happens to be larger than the
patient on frame 0 (poor camera placement, patient seated/lying), you can override the
auto-selection for *those specific videos only* by populating `manual_targets.csv` at the
root of the dataset folder, with format:

```csv
relative_path,bbox_xmin,bbox_ymin,bbox_xmax,bbox_ymax
P3/posture.mp4,120,80,520,710
P7/rest.mp4,200,100,640,700
```

Videos not listed in this CSV use the fully-automatic path. The `_summary.csv` produced
at the end of every run flags the videos most likely to need an override (low detection
rate or suspicious area variability), so in practice the manual CSV either stays empty
or is populated for a small handful of cases.

## Pipeline overview

```
  Clinical .mp4 video
         │
         ▼
  ┌──────────────────────┐
  │ SAM 3 semantic seg.  │   text prompt = 'person'
  │ (per frame)          │   → all detected persons + masks
  └─────────┬────────────┘
            │   (optional second person = neurologist → IGNORED)
            ▼
  ┌──────────────────────┐
  │ Patient selection    │   biggest area + IoU consistency
  │ (single target only) │   with previous frame's patient
  └─────────┬────────────┘
            ▼
  ┌──────────────────────┐
  │ Surface point        │   N contour points (silhouette)
  │ sampling             │   K×L interior-grid points
  └─────────┬────────────┘
            ▼
  ┌──────────────────────┐
  │ Geometric descriptors│   centroid, area, perimeter, bbox,
  │                      │   principal axes, orientation
  └─────────┬────────────┘
            ▼
  outputs/<video_id>_sam3.mp4         ← annotated video
  outputs/<video_id>_sam3_timeseries.xlsx ← per-frame features
```

## Hardware

Tested target: **Windows 11 + NVIDIA RTX GPU with CUDA**.

Note: NVIDIA does not sell an *RTX 3850* — please double-check your card model in
*Device Manager → Display adapters*. Likely candidates are RTX 3050 / 3080 / 4050 / 5050.

VRAM guidance for SAM 3 (`sam3.pt` ≈ 3.45 GB on disk):

- ≥ 12 GB VRAM → run as-is in FP16 (`half=True`), `imgsz=1024`
- 8 GB VRAM → FP16 + `imgsz=768` (default in this notebook)
- 6 GB VRAM → FP16 + `imgsz=640`
- < 6 GB VRAM → fall back to SAM 2 small (see appendix)

If CUDA is not available, the notebook still runs on CPU but expect ~30–60 s per frame for
SAM 3. Plan accordingly.

## 1. Installation

Run these commands **once** in an Anaconda Prompt or PowerShell on Windows 11.
It is strongly recommended to create a fresh conda environment first:

```bat
conda create -n sam3 python=3.11 -y
conda activate sam3
```

Then install PyTorch with CUDA (adjust the CUDA version to your driver — `nvidia-smi`
shows the supported CUDA version):

```bat
pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121
```

Finally install Ultralytics (≥ 8.3.237 is required for SAM 3) and the data libraries:

```bat
pip install -U ultralytics
pip install opencv-python pandas openpyxl matplotlib scipy scikit-image pydantic
```

**Important.** The `sam3.pt` weights are **not** auto-downloaded. You must:

1. Accept the license on Hugging Face: https://huggingface.co/facebook/sam3
2. Download `sam3.pt` (≈ 3.45 GB) and place it in your working directory.

If you later see `TypeError: 'SimpleTokenizer' object is not callable`, fix the wrong
`clip` package with:

```bat
pip uninstall clip -y
pip install git+https://github.com/ultralytics/CLIP.git
```

The cell below is a no-op safeguard you can uncomment if you are running in a notebook
environment instead of using a terminal.

In [ ]:
# Optional in-notebook installation (uncomment if needed). Prefer running these from a
# fresh conda environment in a terminal.

# !pip install -U ultralytics
# !pip install opencv-python pandas openpyxl matplotlib scipy scikit-image pydantic
# !pip uninstall -y clip
# !pip install git+https://github.com/ultralytics/CLIP.git

## 2. Imports and runtime check

In [ ]:
import os
import math
import time
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Optional, Tuple

import numpy as np
import pandas as pd
import cv2

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

from scipy.spatial import distance
from scipy.ndimage import center_of_mass

import torch


# ----------------------------------------------------------------------
# Silence Ultralytics' per-frame WARNING prints (imgsz stride, etc.).
# These messages are benign but produce thousands of lines on long videos,
# which can choke Jupyter's UI and bloat the saved notebook. We keep ERROR
# level so genuine problems still surface.
# ----------------------------------------------------------------------
import logging
logging.getLogger('ultralytics').setLevel(logging.ERROR)

# Ultralytics SAM 3 entry points
#   - SAM3SemanticPredictor      : per-image text/exemplar concept segmentation
#   - SAM3VideoSemanticPredictor : end-to-end video tracking with text prompts
from ultralytics.models.sam import SAM3SemanticPredictor

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch:        {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:            {torch.cuda.get_device_name(0)}')
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'VRAM:           {vram_gb:.1f} GB')
print(f'Device used:    {DEVICE}')

## 3. Configuration

Edit the paths below to match your setup. All paths use raw strings (`r"..."`) to handle
Windows backslashes correctly.

In [ ]:
# ----------------------------------------------------------------------
# Paths (Windows-friendly).
# ----------------------------------------------------------------------
DATASET_DIR = Path(r'C:\Users\<user>\Desktop\sam_3\datasets')
OUTPUT_DIR  = Path(r'C:\Users\<user>\Desktop\sam_3\outputs')

# SAM 3 weights file. Download from https://huggingface.co/facebook/sam3 (~3.45 GB)
MODEL_PATH = Path(r'C:\Users\<user>\Desktop\sam_3\sam3.pt')

# Optional. If this CSV does not exist, the pipeline is 100 % automatic.
MANUAL_TARGETS_CSV = DATASET_DIR / 'manual_targets.csv'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ----------------------------------------------------------------------
# Sampling and inference settings.
# ----------------------------------------------------------------------
@dataclass
class Settings:
    text_prompt:        str   = 'person'   # SAM 3 concept prompt; 'person' is reliable
    confidence:         float = 0.25
    imgsz:              int   = 770        # MUST be a multiple of 14 (SAM 3 backbone stride).
                                       # Useful values: 672, 770, 896, 1008, 1120.
                                       # 1120 is a good choice on >= 12 GB VRAM.
    half:               bool  = True       # FP16 inference (large VRAM saving)

    # Number of points sampled along the silhouette (outer surface).
    n_contour_points:   int   = 64

    # Interior grid resolution: K rows x L columns inside the bbox of the patient mask.
    # Points falling outside the mask are stored as NaN (preserves identity across frames).
    grid_rows:          int   = 12
    grid_cols:          int   = 8          # -> up to 96 interior points

    # Patient selection.
    auto_select_largest:    bool  = True
    iou_continuity_thresh:  float = 0.10

    # Optional manual override applied only on frame 0 of a video. None = fully automatic.
    # When processing a single video, the dispatcher in Section 9 fills this from the
    # manual_targets.csv if a row exists for that video.
    manual_bbox:        Optional[Tuple[int, int, int, int]] = None

    # Sample at most every N-th frame (1 = every frame). Useful to speed up SAM 3.
    frame_stride:       int   = 1

S = Settings()
print(S)

## 4. Discover the dataset (subject-organised, recursive)

Walks `DATASET_DIR` to find videos and parses each parent folder as a subject:

- folder names matching `P<digits>` -> patient,
- folder names matching `C<digits>` -> healthy control,
- anything else is skipped with a warning.

**Filenames and per-subject counts can differ freely.** The notebook does *not* assume
any fixed naming scheme (no hard-coded `rest.mp4` / `posture.mp4` / ...). It picks up
every video file regardless of name or extension and processes them in alphabetical
order. So `P1` may contain three videos called `rest.mp4 / posture.mp4 / action.mp4`,
while `P2` contains a single `IMG_1234.MP4`, while `P3` contains five videos with
arbitrary names, and everything still works.

Supported extensions are configurable (`VIDEO_EXTS`) — by default `.mp4`, `.mov`, `.avi`,
`.mkv` and `.m4v`, all matched case-insensitively.

Also loads the optional `manual_targets.csv` if present, and prints a per-subject summary
showing how many videos were found in each folder and their filenames.

- Subject folder names accept optional suffixes after the integer: `P1`, `P1_RPA`,
  `P12-v2` and `C3_session1` are all valid. The suffix is preserved in
  `subject_id` but ignored for identifying the subject (the integer alone
  determines `subject_num`).


In [ ]:
import re

# Accepted video extensions (case-insensitive). Add more here if needed.
VIDEO_EXTS = {'.mp4', '.mov', '.avi', '.mkv', '.m4v'}

@dataclass
class VideoEntry:
    path:          Path
    relative_path: str            # e.g. 'P3/posture.mp4'
    subject_id:    str            # 'P3', 'C1', ...
    subject_type:  str            # 'patient' / 'control' / 'other'
    subject_num:   Optional[int]  # 3, 1, ...
    manual_bbox:   Optional[Tuple[int, int, int, int]] = None

# Accept folder names like 'P1', 'P1_RPA', 'P12-v2', 'C3', 'C3_session1', etc.
# The mandatory prefix is 'P<n>' (patient) or 'C<n>' (control); anything that
# follows the integer (after an underscore or a dash) is treated as an optional
# descriptive suffix and ignored for identification.
_SUBJECT_RE = re.compile(r'^([PC])(\d+)(?:[_\-].*)?$', re.IGNORECASE)

def parse_subject_folder(name: str) -> Tuple[str, Optional[int]]:
    """Return (subject_type, subject_num) from a folder name.

    Accepted patterns (case-insensitive):
        P1, P12, P1_RPA, P12-v2, P3_session1
        C1, C12, C1_RPA, C12-v2, C3_session1
    The integer right after 'P' or 'C' is the subject number; any '_<suffix>'
    or '-<suffix>' that follows is recorded but does not affect identification.
    """
    m = _SUBJECT_RE.match(name)
    if not m:
        return 'other', None
    prefix, num = m.group(1).upper(), int(m.group(2))
    return ('patient' if prefix == 'P' else 'control'), num

def list_videos_in(folder: Path) -> List[Path]:
    """Case-insensitive multi-extension listing, alphabetically sorted."""
    out = [
        p for p in folder.iterdir()
        if p.is_file()
        and p.suffix.lower() in VIDEO_EXTS
        and not p.name.startswith('.')   # skip hidden / macOS sidecar files
    ]
    return sorted(out, key=lambda p: p.name.lower())

def load_manual_targets(csv_path: Path) -> dict:
    """Return {relative_path: (xmin, ymin, xmax, ymax)} from manual_targets.csv."""
    if not csv_path.exists():
        return {}
    df_t = pd.read_csv(csv_path)
    needed = {'relative_path', 'bbox_xmin', 'bbox_ymin', 'bbox_xmax', 'bbox_ymax'}
    if not needed.issubset(df_t.columns):
        print(f'WARNING: {csv_path} is missing required columns; ignoring it.')
        return {}
    out = {}
    for _, r in df_t.iterrows():
        out[str(r['relative_path']).replace('\\', '/')] = (
            int(r['bbox_xmin']), int(r['bbox_ymin']),
            int(r['bbox_xmax']), int(r['bbox_ymax']),
        )
    return out

def discover_dataset(
    dataset_dir:    Path,
    manual_targets: Optional[dict] = None,
) -> List[VideoEntry]:
    """Walk dataset_dir, parse subject folders, return one VideoEntry per video."""
    if manual_targets is None:
        manual_targets = {}

    entries: List[VideoEntry] = []
    for subject_dir in sorted(p for p in dataset_dir.iterdir() if p.is_dir()):
        subject_id        = subject_dir.name
        subject_type, num = parse_subject_folder(subject_id)
        if subject_type == 'other':
            print(f'WARNING: folder "{subject_id}" does not match P<n>[_suffix] / C<n>[_suffix] -> skipped')
            continue

        # Warn (not crash) if the user has nested subfolders inside a subject folder.
        nested = [p for p in subject_dir.iterdir() if p.is_dir()]
        if nested:
            print(f'WARNING: {subject_dir.name} has subfolder(s) {[n.name for n in nested]} '
                  f'that are NOT scanned (videos must be directly inside {subject_dir.name}/).')

        videos = list_videos_in(subject_dir)
        if not videos:
            print(f'WARNING: no video file in {subject_dir} -> skipped')
            continue
        for v in videos:
            rel = f'{subject_id}/{v.name}'
            entries.append(VideoEntry(
                path          = v,
                relative_path = rel,
                subject_id    = subject_id,
                subject_type  = subject_type,
                subject_num   = num,
                manual_bbox   = manual_targets.get(rel),
            ))
    return entries

def print_dataset_summary(entries: List[VideoEntry]) -> None:
    """Per-subject summary table: count and list of filenames."""
    if not entries:
        print('No videos discovered.'); return

    rows = []
    by_subject = {}
    for e in entries:
        by_subject.setdefault(e.subject_id, []).append(e)

    # Sort subjects: patients first, controls second; within each group, by numeric id.
    _type_priority = {'patient': 0, 'control': 1, 'other': 2}
    def _key(sid):
        es = by_subject[sid]
        return (
            _type_priority.get(es[0].subject_type, 9),
            es[0].subject_num if es[0].subject_num is not None else 10**9,
        )
    subj_order = sorted(by_subject.keys(), key=_key)

    width_id    = max(8,  max(len(s) for s in subj_order))
    width_files = max(40, max(sum(len(e.path.name) + 2 for e in by_subject[s]) for s in subj_order))
    header = f'{"subject":<{width_id}}  {"type":<8}  {"#vids":>5}  filenames'
    print(header)
    print('-' * (len(header) + 20))
    for sid in subj_order:
        es     = by_subject[sid]
        names  = ', '.join(e.path.name for e in es)
        n_man  = sum(e.manual_bbox is not None for e in es)
        suffix = f'  [MANUAL x {n_man}]' if n_man else ''
        print(f'{sid:<{width_id}}  {es[0].subject_type:<8}  {len(es):>5}  {names}{suffix}')

    n_pat  = sum(e.subject_type == 'patient' for e in entries)
    n_ctrl = sum(e.subject_type == 'control' for e in entries)
    n_man  = sum(e.manual_bbox is not None for e in entries)
    print('-' * (len(header) + 20))
    print(f'TOTAL: {len(entries)} videos across {len(by_subject)} subjects '
          f'({n_pat} patient + {n_ctrl} control rows; {n_man} with manual override).')

manual_targets = load_manual_targets(MANUAL_TARGETS_CSV)
video_entries  = discover_dataset(DATASET_DIR, manual_targets)
print_dataset_summary(video_entries)

## 5. Load SAM 3

We use `SAM3SemanticPredictor` because it accepts a *text concept prompt* (`'person'`)
and returns segmentation masks for **all matching instances** in one call. We will then
filter down to the single patient ourselves.

We use the per-image predictor (rather than `SAM3VideoSemanticPredictor`) because we need
fine-grained control over per-frame patient re-identification when a neurologist enters or
leaves the field of view.

In [ ]:
# Load SAM 3 once and reuse the same predictor across all videos and all frames.
overrides = dict(
    conf       = S.confidence,
    task       = 'segment',
    mode       = 'predict',
    model      = str(MODEL_PATH),
    half       = S.half,
    imgsz      = S.imgsz,
    save       = False,        # we draw and save annotations ourselves
    verbose    = False,
    device     = 0 if DEVICE == 'cuda' else 'cpu',
)

predictor = SAM3SemanticPredictor(overrides=overrides)
print('SAM 3 predictor ready.')

## 6. Helper functions

### 6.1. Selecting the patient and ignoring everyone else

When two people are on screen (typically the patient and a neurologist), SAM 3 returns one
mask per person. We need to lock onto **one** of them — the patient — and ignore the rest
for the entire video.

Strategy:

- **Frame 0** — pick the largest mask (the patient is usually closer to the camera and
  more centrally framed), unless the user supplies a manual bounding box.
- **Subsequent frames** — among the candidate masks, pick the one with the highest IoU
  with the previous frame's patient mask. This keeps the identity stable even if the
  patient temporarily becomes smaller than the neurologist.
- If no mask is reasonably consistent (IoU below threshold), we fall back to the
  closest-centroid candidate, and as a last resort we mark the frame as missing (NaN row).

In [ ]:
def mask_iou(a: np.ndarray, b: np.ndarray) -> float:
    """Intersection-over-Union between two boolean masks of the same shape."""
    if a is None or b is None:
        return 0.0
    inter = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    return float(inter / union) if union > 0 else 0.0

def mask_centroid(mask: np.ndarray) -> Tuple[float, float]:
    """Return the (x, y) centroid of a boolean mask. NaN if mask is empty."""
    if mask.sum() == 0:
        return (float('nan'), float('nan'))
    cy, cx = center_of_mass(mask.astype(np.uint8))
    return float(cx), float(cy)

def select_patient_mask(
    masks:        np.ndarray,             # shape (K, H, W) bool
    boxes:        np.ndarray,             # shape (K, 4) xyxy
    prev_mask:    Optional[np.ndarray],   # last frame's patient mask, or None
    cfg:          Settings,
) -> Tuple[Optional[np.ndarray], Optional[np.ndarray]]:
    """
    Choose ONE mask among the K candidates returned by SAM 3.

    Returns
    -------
    mask_sel : (H, W) bool, or None if no plausible patient was found
    box_sel  : (4,) np.ndarray xyxy, or None
    """
    if masks is None or len(masks) == 0:
        return None, None

    # Cast to bool
    masks = masks.astype(bool)

    # ------------------------------------------------------------------
    # FRAME 0 case (no previous mask)
    # ------------------------------------------------------------------
    if prev_mask is None:
        # Manual bbox override?
        if cfg.manual_bbox is not None:
            xmin, ymin, xmax, ymax = cfg.manual_bbox
            best_idx, best_iou = -1, -1
            for i, b in enumerate(boxes):
                # IoU between cfg.manual_bbox and candidate box
                xa = max(b[0], xmin); ya = max(b[1], ymin)
                xb = min(b[2], xmax); yb = min(b[3], ymax)
                inter = max(0, xb - xa) * max(0, yb - ya)
                area_a = (b[2] - b[0]) * (b[3] - b[1])
                area_b = (xmax - xmin) * (ymax - ymin)
                union = area_a + area_b - inter
                iou = inter / union if union > 0 else 0
                if iou > best_iou:
                    best_iou, best_idx = iou, i
            return masks[best_idx], boxes[best_idx]

        # Default: largest area
        if cfg.auto_select_largest:
            areas = masks.reshape(len(masks), -1).sum(axis=1)
            idx = int(np.argmax(areas))
            return masks[idx], boxes[idx]

    # ------------------------------------------------------------------
    # SUBSEQUENT FRAMES: re-identify by IoU with the previous mask
    # ------------------------------------------------------------------
    ious = np.array([mask_iou(m, prev_mask) for m in masks])
    best = int(np.argmax(ious))
    if ious[best] >= cfg.iou_continuity_thresh:
        return masks[best], boxes[best]

    # Fallback: closest centroid to the previous patient's centroid.
    pcx, pcy = mask_centroid(prev_mask)
    if not math.isnan(pcx):
        dists = []
        for m in masks:
            cx, cy = mask_centroid(m)
            if math.isnan(cx):
                dists.append(np.inf)
            else:
                dists.append((cx - pcx) ** 2 + (cy - pcy) ** 2)
        idx = int(np.argmin(dists))
        # Sanity check: also require non-trivial overlap with previous bbox area
        if not np.isinf(dists[idx]):
            return masks[idx], boxes[idx]

    return None, None

### 6.2. Sampling points on the body surface

From a binary mask of the patient we extract two complementary point clouds:

1. **Outer surface (silhouette)** — the largest external contour, then resampled to
   exactly `n_contour_points` evenly spaced points (by perimeter arc-length).
   The starting point is the contour vertex closest to the centroid + (0, -1) direction,
   i.e. roughly above the centroid (head region for a standing/sitting person).
   This anchoring gives reasonable correspondence between consecutive frames.
2. **Inner surface (interior)** — a regular `K × L` grid covering the bounding box of the
   mask. Each grid node `(i, j)` is identified by its normalized position
   `(i / (K - 1), j / (L - 1))` inside the bbox, so it has the *same semantic identity*
   across frames (e.g. *(0.5, 0.5) = body centre*). Nodes falling outside the mask are
   stored as `NaN`.

In [ ]:
def get_main_contour(mask: np.ndarray) -> Optional[np.ndarray]:
    """Return the largest external contour (N, 2) in (x, y), or None if mask is empty."""
    cnts, _ = cv2.findContours(
        mask.astype(np.uint8) * 255,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_NONE,
    )
    if not cnts:
        return None
    main = max(cnts, key=cv2.contourArea)
    pts  = main.reshape(-1, 2).astype(np.float32)   # (N, 2)
    return pts

def resample_contour(pts: np.ndarray, n: int, anchor_xy: Tuple[float, float]) -> np.ndarray:
    """
    Resample a contour to exactly `n` points evenly spaced by perimeter arc-length.

    The first output point is the contour vertex closest to `anchor_xy`. This stabilizes
    the index ordering across frames (so 'point 0' is consistently the same body region).
    """
    # 1) Reorder the contour so it starts at the vertex closest to the anchor.
    diffs   = pts - np.array(anchor_xy, dtype=np.float32)
    start   = int(np.argmin((diffs ** 2).sum(axis=1)))
    pts     = np.roll(pts, -start, axis=0)

    # 2) Compute cumulative arc length around the contour.
    seg     = np.linalg.norm(np.diff(pts, axis=0, append=pts[:1]), axis=1)
    cumlen  = np.concatenate(([0.0], np.cumsum(seg[:-1])))
    total   = cumlen[-1] + seg[-1]
    if total <= 1e-6:
        return np.repeat(pts[:1], n, axis=0)

    # 3) Sample n target lengths evenly along the perimeter.
    targets = np.linspace(0, total, n, endpoint=False)
    out     = np.zeros((n, 2), dtype=np.float32)
    for k, t in enumerate(targets):
        idx = int(np.searchsorted(cumlen, t, side='right') - 1)
        idx = max(0, min(idx, len(pts) - 2))
        seg_len = seg[idx] if seg[idx] > 1e-6 else 1.0
        alpha   = (t - cumlen[idx]) / seg_len
        out[k]  = pts[idx] * (1 - alpha) + pts[(idx + 1) % len(pts)] * alpha
    return out

def sample_contour_points(mask: np.ndarray, n: int) -> np.ndarray:
    """Top-level: return n evenly-spaced contour points (n, 2). NaN if mask empty."""
    pts = get_main_contour(mask)
    if pts is None or len(pts) < 3:
        return np.full((n, 2), np.nan, dtype=np.float32)

    cx, cy = mask_centroid(mask)
    # Anchor = highest point along the contour above the centroid (≈ head)
    above = pts[pts[:, 1] < cy]
    if len(above) > 0:
        anchor_idx = int(np.argmin(above[:, 1]))
        anchor = (float(above[anchor_idx, 0]), float(above[anchor_idx, 1]))
    else:
        anchor = (cx, cy)
    return resample_contour(pts, n, anchor)

def sample_interior_grid(
    mask: np.ndarray,
    box:  np.ndarray,           # xyxy
    rows: int,
    cols: int,
) -> np.ndarray:
    """
    Sample a `rows × cols` grid inside the bbox of the mask.

    Returns
    -------
    out : (rows*cols, 2) float32
        Grid in row-major order (top→bottom, left→right). Points outside the mask = NaN.
    """
    H, W = mask.shape
    x1, y1, x2, y2 = [float(v) for v in box]

    out = np.full((rows * cols, 2), np.nan, dtype=np.float32)
    if x2 <= x1 or y2 <= y1:
        return out

    for r in range(rows):
        v = r / max(rows - 1, 1)
        py = y1 + v * (y2 - y1)
        for c in range(cols):
            u = c / max(cols - 1, 1)
            px = x1 + u * (x2 - x1)
            ix = int(round(px)); iy = int(round(py))
            if 0 <= ix < W and 0 <= iy < H and mask[iy, ix]:
                out[r * cols + c] = (px, py)
    return out

### 6.3. Geometric descriptors

For each frame we compute a small set of clinically interpretable descriptors that
summarize the patient's body silhouette. These are useful as standalone features for
downstream movement-disorder classification with TabICLv2.

In [ ]:
def geometric_descriptors(mask: np.ndarray, box: np.ndarray) -> dict:
    """
    Return a dict of geometric descriptors for the body silhouette:
      - centroid_x, centroid_y
      - area_px                (number of mask pixels)
      - perimeter_px           (length of the main contour)
      - bbox_x, bbox_y, bbox_w, bbox_h
      - aspect_ratio           (h / w)
      - solidity               (area / convex_hull_area)
      - extent                 (area / bbox_area)
      - orientation_deg        (angle of principal axis, in [-90, +90])
      - major_axis_px          (length of major axis from PCA)
      - minor_axis_px          (length of minor axis from PCA)
    All values are NaN when the mask is empty."""
    out = {
        'centroid_x': np.nan, 'centroid_y': np.nan,
        'area_px': np.nan, 'perimeter_px': np.nan,
        'bbox_x': np.nan, 'bbox_y': np.nan, 'bbox_w': np.nan, 'bbox_h': np.nan,
        'aspect_ratio': np.nan, 'solidity': np.nan, 'extent': np.nan,
        'orientation_deg': np.nan, 'major_axis_px': np.nan, 'minor_axis_px': np.nan,
    }
    if mask is None or mask.sum() == 0:
        return out

    cx, cy = mask_centroid(mask)
    out['centroid_x'] = cx; out['centroid_y'] = cy
    out['area_px'] = float(mask.sum())

    cnts, _ = cv2.findContours(
        mask.astype(np.uint8) * 255, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE
    )
    if not cnts:
        return out
    main = max(cnts, key=cv2.contourArea)
    out['perimeter_px'] = float(cv2.arcLength(main, True))

    x1, y1, x2, y2 = [float(v) for v in box]
    out['bbox_x'], out['bbox_y'] = x1, y1
    out['bbox_w'], out['bbox_h'] = x2 - x1, y2 - y1
    if (x2 - x1) > 0:
        out['aspect_ratio'] = (y2 - y1) / (x2 - x1)
        out['extent'] = out['area_px'] / max((x2 - x1) * (y2 - y1), 1)

    hull = cv2.convexHull(main)
    hull_area = cv2.contourArea(hull)
    if hull_area > 0:
        out['solidity'] = out['area_px'] / hull_area

    # Principal axes via PCA on the mask pixels (subsample for speed if very large)
    ys, xs = np.where(mask)
    if len(xs) >= 5:
        if len(xs) > 5000:
            idx = np.random.default_rng(0).choice(len(xs), size=5000, replace=False)
            xs, ys = xs[idx], ys[idx]
        pts = np.stack([xs, ys], axis=1).astype(np.float32)
        pts -= pts.mean(axis=0, keepdims=True)
        cov = np.cov(pts.T)
        evals, evecs = np.linalg.eigh(cov)
        order = np.argsort(evals)[::-1]
        evals, evecs = evals[order], evecs[:, order]
        major_vec = evecs[:, 0]
        out['orientation_deg'] = float(np.degrees(np.arctan2(major_vec[1], major_vec[0])))
        # Convert eigenvalues (variance) to lengths (≈ 4σ covers ~95 % of points)
        out['major_axis_px'] = float(4.0 * np.sqrt(max(evals[0], 0)))
        out['minor_axis_px'] = float(4.0 * np.sqrt(max(evals[1], 0)))
    return out

### 6.4. Per-frame SAM 3 inference wrapper

Small wrapper around `SAM3SemanticPredictor` that accepts an OpenCV BGR frame, runs
concept segmentation with `text='person'`, and returns `(masks, boxes)` as NumPy arrays.

In [ ]:
import tempfile

def _set_image_robust(predictor, frame_bgr: np.ndarray):
    """Call SAM 3's set_image() whether it expects a path or a numpy array."""
    try:
        predictor.set_image(frame_bgr)            # ndarray path (preferred, fast)
        return None
    except Exception:
        # Fallback: save to a temp file and pass the path.
        tmp = tempfile.NamedTemporaryFile(suffix='.png', delete=False)
        tmp.close()
        cv2.imwrite(tmp.name, frame_bgr)
        predictor.set_image(tmp.name)
        return tmp.name                           # caller is responsible for cleanup

def sam3_segment_persons(
    frame_bgr: np.ndarray,
    predictor: SAM3SemanticPredictor,
    text:      str = 'person',
) -> Tuple[np.ndarray, np.ndarray]:
    """Run SAM 3 semantic segmentation on a single BGR frame.

    Returns
    -------
    masks : (K, H, W) bool
        One boolean mask per detected instance. Empty array if nothing detected.
    boxes : (K, 4) float32
        xyxy bounding boxes in pixel coordinates of the original frame.
    """
    H, W = frame_bgr.shape[:2]

    tmp_path = _set_image_robust(predictor, frame_bgr)
    try:
        results = predictor(text=[text])
    finally:
        if tmp_path is not None:
            try: os.unlink(tmp_path)
            except Exception: pass

    # Ultralytics may return a list of Results or a single Results object.
    if isinstance(results, list):
        if len(results) == 0:
            return np.zeros((0, H, W), dtype=bool), np.zeros((0, 4), dtype=np.float32)
        r = results[0]
    else:
        r = results

    if r is None or r.masks is None or r.masks.data is None or len(r.masks.data) == 0:
        return np.zeros((0, H, W), dtype=bool), np.zeros((0, 4), dtype=np.float32)

    # masks.data is a torch tensor of shape (K, h, w) at the model's processed resolution.
    masks_t = r.masks.data.detach().cpu().numpy().astype(bool)
    if masks_t.shape[1:] != (H, W):
        # Resize each mask to the original frame size with nearest-neighbour.
        masks_resized = np.zeros((masks_t.shape[0], H, W), dtype=bool)
        for i in range(masks_t.shape[0]):
            mr = cv2.resize(
                masks_t[i].astype(np.uint8), (W, H),
                interpolation=cv2.INTER_NEAREST,
            )
            masks_resized[i] = mr.astype(bool)
        masks = masks_resized
    else:
        masks = masks_t

    if r.boxes is not None and r.boxes.xyxy is not None and len(r.boxes.xyxy) > 0:
        boxes = r.boxes.xyxy.detach().cpu().numpy().astype(np.float32)
    else:
        # Recompute boxes from masks if SAM 3 didn't return them.
        boxes = np.zeros((len(masks), 4), dtype=np.float32)
        for i, m in enumerate(masks):
            ys, xs = np.where(m)
            if len(xs) > 0:
                boxes[i] = [xs.min(), ys.min(), xs.max(), ys.max()]
    return masks, boxes

### 6.5. Annotation drawing — three complementary video outputs

Per processed video, three writers run in parallel and produce three different views of
the same patient segmentation:

1. **`draw_annotation`** — original frame + magenta-tinted patient mask + green bbox +
   contour points (orange) + interior grid points (cyan) + centroid (red cross).
   Other detected persons are simply *not drawn* — they never appear on the output.
2. **`render_mask_only`** — black background, white silhouette of the patient with a
   magenta outline. Useful for downstream silhouette-only processing.
3. **`render_sidebyside`** — `original | mask | overlay` concatenated horizontally,
   ideal for clinical inspection at a glance.

In [ ]:
def draw_annotation(
    frame_bgr:  np.ndarray,
    mask:       Optional[np.ndarray],
    box:        Optional[np.ndarray],
    contour_pts: Optional[np.ndarray],   # (N, 2)
    grid_pts:    Optional[np.ndarray],   # (M, 2), NaN for outside-mask
    descr:       Optional[dict],
) -> np.ndarray:
    """Return a BGR copy of `frame_bgr` overlaid with patient annotations."""
    out = frame_bgr.copy()
    if mask is not None and mask.any():
        # Translucent magenta tint of the mask
        tint = np.zeros_like(out)
        tint[mask] = (180, 50, 220)        # BGR: pinkish magenta
        out = cv2.addWeighted(out, 1.0, tint, 0.35, 0)

        # Mask contour (white, thin)
        cnts, _ = cv2.findContours(
            mask.astype(np.uint8) * 255, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE
        )
        if cnts:
            cv2.drawContours(out, cnts, -1, (255, 255, 255), 1, cv2.LINE_AA)

    if box is not None:
        x1, y1, x2, y2 = [int(v) for v in box]
        cv2.rectangle(out, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(out, 'patient', (x1, max(0, y1 - 6)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1, cv2.LINE_AA)

    if grid_pts is not None:
        for x, y in grid_pts:
            if not np.isnan(x):
                cv2.circle(out, (int(x), int(y)), 2, (255, 220, 0), -1, cv2.LINE_AA)

    if contour_pts is not None:
        for x, y in contour_pts:
            if not np.isnan(x):
                cv2.circle(out, (int(x), int(y)), 3, (0, 140, 255), -1, cv2.LINE_AA)

    if descr is not None and not np.isnan(descr.get('centroid_x', np.nan)):
        cx, cy = int(descr['centroid_x']), int(descr['centroid_y'])
        cv2.drawMarker(out, (cx, cy), (0, 0, 255), cv2.MARKER_CROSS, 16, 2, cv2.LINE_AA)
    return out

def render_mask_only(
    frame_shape: Tuple[int, int],   # (H, W)
    mask:        Optional[np.ndarray],
    contour_pts: Optional[np.ndarray] = None,
) -> np.ndarray:
    """Black background, white-filled patient silhouette + magenta outline.

    A pure-mask video is convenient for downstream image processing tasks (e.g. computing
    optical flow only inside the patient, or training a model on silhouettes only)."""
    H, W = frame_shape
    out = np.zeros((H, W, 3), dtype=np.uint8)
    if mask is not None and mask.any():
        out[mask] = (240, 240, 240)        # near-white fill
        cnts, _ = cv2.findContours(
            mask.astype(np.uint8) * 255, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE
        )
        if cnts:
            cv2.drawContours(out, cnts, -1, (180, 50, 220), 2, cv2.LINE_AA)
        if contour_pts is not None:
            for x, y in contour_pts:
                if not np.isnan(x):
                    cv2.circle(out, (int(x), int(y)), 3, (0, 140, 255), -1, cv2.LINE_AA)
    return out

def render_sidebyside(
    raw_bgr:    np.ndarray,
    mask_bgr:   np.ndarray,
    overlay_bgr: np.ndarray,
    labels:     Tuple[str, str, str] = ('original', 'mask', 'overlay'),
) -> np.ndarray:
    """Concatenate the three frames horizontally with text labels on top."""
    H, W = raw_bgr.shape[:2]
    panels = [raw_bgr, mask_bgr, overlay_bgr]
    out = np.concatenate(panels, axis=1)
    for i, label in enumerate(labels):
        cv2.putText(out, label, (i * W + 10, 24),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2, cv2.LINE_AA)
        cv2.line(out, (i * W, 0), (i * W, H), (60, 60, 60), 1)
    return out

### 6.6. Output schema

Each row of the output Excel corresponds to one processed frame. Columns are:

- `time_s`, `frame_idx`, `patient_detected`
- 14 geometric descriptors (centroid, area, …)
- `2 × n_contour_points` columns: `contour_<i>_x`, `contour_<i>_y`
- `2 × (rows × cols)` columns: `grid_r<r>_c<c>_x`, `grid_r<r>_c<c>_y`

This is the table that gets fed to **TabICLv2**.

In [ ]:
GEOM_KEYS = [
    'centroid_x', 'centroid_y', 'area_px', 'perimeter_px',
    'bbox_x', 'bbox_y', 'bbox_w', 'bbox_h',
    'aspect_ratio', 'solidity', 'extent',
    'orientation_deg', 'major_axis_px', 'minor_axis_px',
]

META_KEYS = ['subject_id', 'subject_type', 'video_relpath']

def build_columns(cfg: Settings) -> List[str]:
    cols = list(META_KEYS) + ['time_s', 'frame_idx', 'patient_detected']
    cols += GEOM_KEYS
    for i in range(cfg.n_contour_points):
        cols += [f'contour_{i:03d}_x', f'contour_{i:03d}_y']
    for r in range(cfg.grid_rows):
        for c in range(cfg.grid_cols):
            cols += [f'grid_r{r:02d}_c{c:02d}_x', f'grid_r{r:02d}_c{c:02d}_y']
    return cols

def empty_row(
    cfg:           Settings,
    time_s:        float,
    frame_idx:     int,
    subject_id:    str = '',
    subject_type:  str = '',
    video_relpath: str = '',
) -> dict:
    row = {
        'subject_id':    subject_id,
        'subject_type':  subject_type,
        'video_relpath': video_relpath,
        'time_s':        time_s,
        'frame_idx':     frame_idx,
        'patient_detected': 0,
    }
    for k in GEOM_KEYS:
        row[k] = np.nan
    for i in range(cfg.n_contour_points):
        row[f'contour_{i:03d}_x'] = np.nan
        row[f'contour_{i:03d}_y'] = np.nan
    for r in range(cfg.grid_rows):
        for c in range(cfg.grid_cols):
            row[f'grid_r{r:02d}_c{c:02d}_x'] = np.nan
            row[f'grid_r{r:02d}_c{c:02d}_y'] = np.nan
    return row

## 7. Main per-video pipeline

Drop-in replacement for the `member_tremor()` function of the YOLO notebook. Same input
(`file_name`) and same output convention (`<video_id>_sam3.mp4` + `<video_id>_sam3_timeseries.xlsx`).

In [ ]:
def process_video_sam3(
    entry:      VideoEntry,
    predictor:  SAM3SemanticPredictor,
    cfg:        Settings,
    output_dir: Path,
    write_overlay:     bool = True,
    write_mask:        bool = True,
    write_sidebyside:  bool = True,
) -> pd.DataFrame:
    """Process one video end-to-end with SAM 3 + dense surface sampling.

    Outputs go into `output_dir/<subject_id>/`. Filenames mirror the input:
      <output_dir>/<subject>/<stem>_sam3_overlay.mp4      annotated overlay
      <output_dir>/<subject>/<stem>_sam3_mask.mp4         silhouette only
      <output_dir>/<subject>/<stem>_sam3_sidebyside.mp4   raw | mask | overlay
      <output_dir>/<subject>/<stem>_sam3_timeseries.xlsx  per-frame features

    The Excel includes columns subject_id / subject_type / video_relpath so that all
    per-video files concatenate cleanly into a single TabICLv2-ready table.
    """
    file_name = entry.path
    cap = cv2.VideoCapture(str(file_name))
    if not cap.isOpened():
        raise RuntimeError(f'Cannot open {file_name}')

    fps     = cap.get(cv2.CAP_PROP_FPS) or 30.0
    width   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    nframes = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    stem        = file_name.stem
    subject_dir = output_dir / entry.subject_id
    subject_dir.mkdir(parents=True, exist_ok=True)
    out_overlay     = subject_dir / f'{stem}_sam3_overlay.mp4'
    out_mask        = subject_dir / f'{stem}_sam3_mask.mp4'
    out_sidebyside  = subject_dir / f'{stem}_sam3_sidebyside.mp4'
    out_excel_path  = subject_dir / f'{stem}_sam3_timeseries.xlsx'

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer_overlay = (
        cv2.VideoWriter(str(out_overlay), fourcc, fps, (width, height))
        if write_overlay else None
    )
    writer_mask = (
        cv2.VideoWriter(str(out_mask), fourcc, fps, (width, height))
        if write_mask else None
    )
    writer_sbs = (
        cv2.VideoWriter(str(out_sidebyside), fourcc, fps, (width * 3, height))
        if write_sidebyside else None
    )

    # Apply manual override on this video, if present in the CSV.
    cfg_local = Settings(**{**cfg.__dict__, 'manual_bbox': entry.manual_bbox})

    cols = build_columns(cfg_local)
    rows: List[dict] = []

    prev_mask: Optional[np.ndarray] = None
    frame_idx = -1
    t0 = time.time()

    target_mode = 'MANUAL' if entry.manual_bbox is not None else 'auto'
    print(f'[{entry.subject_id}/{stem}] {nframes} frames, {fps:.1f} fps, '
          f'{width}x{height}  target={target_mode} -> start')

    while True:
        ok, frame = cap.read()
        if not ok:
            break
        frame_idx += 1
        time_s = round(cap.get(cv2.CAP_PROP_POS_MSEC) / 1000.0, 3)

        if cfg_local.frame_stride > 1 and (frame_idx % cfg_local.frame_stride) != 0:
            continue

        # ---- 1) SAM 3 segmentation ----
        masks, boxes = sam3_segment_persons(frame, predictor, cfg_local.text_prompt)

        # ---- 2) patient selection (ignores all other persons) ----
        mask_sel, box_sel = select_patient_mask(masks, boxes, prev_mask, cfg_local)

        meta = dict(
            subject_id    = entry.subject_id,
            subject_type  = entry.subject_type,
            video_relpath = entry.relative_path,
        )

        if mask_sel is None or not mask_sel.any():
            row = empty_row(cfg_local, time_s, frame_idx, **meta)
            rows.append(row)
            empty_mask_frame = np.zeros_like(frame)
            if writer_overlay is not None: writer_overlay.write(frame)
            if writer_mask    is not None: writer_mask.write(empty_mask_frame)
            if writer_sbs     is not None:
                writer_sbs.write(render_sidebyside(frame, empty_mask_frame, frame))
            continue

        prev_mask = mask_sel

        # ---- 3) sampling: contour + interior grid ----
        contour_pts = sample_contour_points(mask_sel, cfg_local.n_contour_points)
        grid_pts    = sample_interior_grid(
            mask_sel, box_sel, cfg_local.grid_rows, cfg_local.grid_cols
        )

        # ---- 4) geometric descriptors ----
        descr = geometric_descriptors(mask_sel, box_sel)

        # ---- 5) build row ----
        row = empty_row(cfg_local, time_s, frame_idx, **meta)
        row['patient_detected'] = 1
        for k in GEOM_KEYS:
            row[k] = descr[k]
        for i, (x, y) in enumerate(contour_pts):
            row[f'contour_{i:03d}_x'] = float(x) if not np.isnan(x) else np.nan
            row[f'contour_{i:03d}_y'] = float(y) if not np.isnan(y) else np.nan
        idx = 0
        for r in range(cfg_local.grid_rows):
            for c in range(cfg_local.grid_cols):
                gx, gy = grid_pts[idx]
                row[f'grid_r{r:02d}_c{c:02d}_x'] = float(gx) if not np.isnan(gx) else np.nan
                row[f'grid_r{r:02d}_c{c:02d}_y'] = float(gy) if not np.isnan(gy) else np.nan
                idx += 1
        rows.append(row)

        # ---- 6) annotated frames (three views) ----
        overlay_frame = draw_annotation(frame, mask_sel, box_sel, contour_pts, grid_pts, descr)
        mask_frame    = render_mask_only((height, width), mask_sel, contour_pts)
        if writer_overlay is not None: writer_overlay.write(overlay_frame)
        if writer_mask    is not None: writer_mask.write(mask_frame)
        if writer_sbs     is not None:
            writer_sbs.write(render_sidebyside(frame, mask_frame, overlay_frame))

        if frame_idx % 50 == 0 and frame_idx > 0:
            elapsed = time.time() - t0
            rate    = (frame_idx + 1) / elapsed
            eta     = (nframes - frame_idx - 1) / max(rate, 1e-6)
            print(f'  [{frame_idx:5d}/{nframes}]  {rate:.2f} fps  ETA {eta/60:.1f} min')

    cap.release()
    if writer_overlay is not None: writer_overlay.release()
    if writer_mask    is not None: writer_mask.release()
    if writer_sbs     is not None: writer_sbs.release()

    df = pd.DataFrame(rows, columns=cols)
    df.to_excel(out_excel_path, index=False)

    n_det = int(df.patient_detected.sum())
    print(f'[{entry.subject_id}/{stem}] done in {(time.time() - t0)/60:.1f} min  '
          f'(detected on {n_det}/{len(df)} frames)')
    return df

## 8. Visualisations

### 8.1. Sanity-check visualisation on one frame of one video

Useful before launching the full batch. The figure below shows, for a chosen video:

- the raw frame,
- the binary patient mask (other persons are *not* drawn),
- the silhouette + interior point clouds,
- the final annotated frame as it is written to the output video.

In [ ]:
def quicklook(
    entry:     VideoEntry,
    predictor: SAM3SemanticPredictor,
    cfg:       Settings,
    frame_idx: int = 0,
):
    cap = cv2.VideoCapture(str(entry.path))
    if not cap.isOpened():
        raise RuntimeError(f'Cannot open {entry.path}')
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ok, frame = cap.read()
    cap.release()
    if not ok:
        raise RuntimeError(f'Frame {frame_idx} unreadable in {entry.path}')

    cfg_local = Settings(**{**cfg.__dict__, 'manual_bbox': entry.manual_bbox})
    masks, boxes = sam3_segment_persons(frame, predictor, cfg_local.text_prompt)
    mask, box    = select_patient_mask(masks, boxes, None, cfg_local)
    if mask is None:
        print('No patient detected on this frame.')
        plt.figure(figsize=(8, 5))
        plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)); plt.axis('off'); plt.show()
        return

    contour_pts = sample_contour_points(mask, cfg_local.n_contour_points)
    grid_pts    = sample_interior_grid(mask, box, cfg_local.grid_rows, cfg_local.grid_cols)
    descr       = geometric_descriptors(mask, box)
    annotated   = draw_annotation(frame, mask, box, contour_pts, grid_pts, descr)

    fig, ax = plt.subplots(2, 2, figsize=(13, 9))
    ax[0, 0].imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    ax[0, 0].set_title(f'Raw frame {frame_idx}'); ax[0, 0].axis('off')

    ax[0, 1].imshow(mask, cmap='magma')
    n_persons = len(masks)
    extra = f' (out of {n_persons} detected person(s) - {n_persons-1} ignored)' if n_persons > 1 else ''
    ax[0, 1].set_title(f'Patient binary mask{extra}'); ax[0, 1].axis('off')

    ax[1, 0].imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    if not np.isnan(contour_pts).all():
        ax[1, 0].scatter(contour_pts[:, 0], contour_pts[:, 1],
                         s=20, c='orange', edgecolors='black', linewidths=0.4,
                         label='outer (silhouette)')
    if not np.isnan(grid_pts).all():
        ok_pts = ~np.isnan(grid_pts[:, 0])
        ax[1, 0].scatter(grid_pts[ok_pts, 0], grid_pts[ok_pts, 1],
                         s=10, c='cyan', edgecolors='black', linewidths=0.3,
                         label='inner (grid)')
    ax[1, 0].set_title(f'Surface point cloud  ({(~np.isnan(contour_pts[:,0])).sum()} outer + '
                       f'{(~np.isnan(grid_pts[:,0])).sum()} inner)')
    ax[1, 0].axis('off'); ax[1, 0].legend(loc='lower right', fontsize=8)

    ax[1, 1].imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    ax[1, 1].set_title('Final overlay (as written to output video)'); ax[1, 1].axis('off')

    plt.suptitle(f'SAM 3 quicklook  -  {entry.relative_path}', fontsize=12)
    plt.tight_layout()
    plt.show()

# Example: change the index to test on different videos / frames.
if len(video_entries) > 0:
    quicklook(video_entries[0], predictor, S, frame_idx=0)

### 8.2. Temporal visualisation of the extracted features

After processing a video, this helper plots:

- the centroid trajectory in image space,
- area, perimeter and orientation as time series,
- a heatmap of (x, y) variance for every interior grid point — this immediately reveals
  which body regions move the most during the recording, and is a useful clinical sanity
  check for tremor, chorea, dystonia, etc.

In [ ]:
def plot_video_features(df: pd.DataFrame, cfg: Settings, video_id: str = ''):
    valid = df[df['patient_detected'] == 1]
    if len(valid) == 0:
        print('No patient frames in this video.')
        return

    fig = plt.figure(figsize=(13, 9))
    gs  = fig.add_gridspec(3, 3, hspace=0.4, wspace=0.3)

    # ---- centroid trajectory ----
    ax = fig.add_subplot(gs[0, 0])
    ax.plot(valid['centroid_x'], valid['centroid_y'],
            color='red', alpha=0.6, linewidth=1)
    ax.scatter(valid['centroid_x'].iloc[0], valid['centroid_y'].iloc[0],
               c='green', s=40, marker='o', label='start', zorder=5)
    ax.scatter(valid['centroid_x'].iloc[-1], valid['centroid_y'].iloc[-1],
               c='black', s=40, marker='X', label='end', zorder=5)
    ax.invert_yaxis()  # image y axis goes down
    ax.set_xlabel('x (px)'); ax.set_ylabel('y (px)')
    ax.set_title('Centroid trajectory'); ax.legend(fontsize=8); ax.grid(alpha=0.3)

    # ---- area ----
    ax = fig.add_subplot(gs[0, 1])
    ax.plot(valid['time_s'], valid['area_px'], color='steelblue')
    ax.set_xlabel('time (s)'); ax.set_ylabel('area (px)')
    ax.set_title('Body silhouette area'); ax.grid(alpha=0.3)

    # ---- orientation ----
    ax = fig.add_subplot(gs[0, 2])
    ax.plot(valid['time_s'], valid['orientation_deg'], color='darkorange')
    ax.set_xlabel('time (s)'); ax.set_ylabel('angle (deg)')
    ax.set_title('Principal-axis orientation'); ax.grid(alpha=0.3)

    # ---- perimeter and aspect ratio ----
    ax = fig.add_subplot(gs[1, 0])
    ax.plot(valid['time_s'], valid['perimeter_px'], color='purple')
    ax.set_xlabel('time (s)'); ax.set_ylabel('perimeter (px)')
    ax.set_title('Silhouette perimeter'); ax.grid(alpha=0.3)

    ax = fig.add_subplot(gs[1, 1])
    ax.plot(valid['time_s'], valid['aspect_ratio'], color='teal')
    ax.set_xlabel('time (s)'); ax.set_ylabel('h / w')
    ax.set_title('Bounding-box aspect ratio'); ax.grid(alpha=0.3)

    ax = fig.add_subplot(gs[1, 2])
    ax.plot(valid['time_s'], valid['solidity'], color='brown')
    ax.set_xlabel('time (s)'); ax.set_ylabel('solidity')
    ax.set_title('Silhouette solidity'); ax.grid(alpha=0.3)

    # ---- per-grid-point motion variance heatmap ----
    ax = fig.add_subplot(gs[2, :])
    var_map = np.full((cfg.grid_rows, cfg.grid_cols), np.nan)
    for r in range(cfg.grid_rows):
        for c in range(cfg.grid_cols):
            xs = valid[f'grid_r{r:02d}_c{c:02d}_x']
            ys = valid[f'grid_r{r:02d}_c{c:02d}_y']
            if xs.notna().sum() > 5:
                var_map[r, c] = float(np.nanvar(xs) + np.nanvar(ys))
    im = ax.imshow(var_map, cmap='hot', aspect='auto')
    plt.colorbar(im, ax=ax, label='spatial variance (px²)')
    ax.set_xlabel('grid column'); ax.set_ylabel('grid row')
    ax.set_title('Where on the body does the motion happen?  '
                 '(per interior-grid-point variance over time)')

    fig.suptitle(f'Per-video feature summary  ·  {video_id}', fontsize=13)
    plt.show()

### 8.3. 3D space-time view of the surface point cloud

Plotting the contour and grid points in a 3D scatter where the **Z axis is time** turns the time series into a static figure that is very useful for spotting periodic motion (tremor) or chaotic motion (chorea, myoclonus) by eye.

Each contour point becomes a coloured *strand* through time; the strand's wiggle is the phenomenology.

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (registers the 3D projection)

def plot_surface_spacetime(
    df:        pd.DataFrame,
    cfg:       Settings,
    video_id:  str = '',
    show_grid: bool = True,
    show_contour: bool = True,
    max_strands: int = 24,
):
    """3D scatter where Z = time, X/Y = pixel position of each surface point."""
    valid = df[df['patient_detected'] == 1].copy()
    if len(valid) == 0:
        print('No patient frames in this video.')
        return

    fig = plt.figure(figsize=(11, 8))
    ax  = fig.add_subplot(111, projection='3d')

    t = valid['time_s'].to_numpy()
    cmap = plt.cm.viridis

    if show_contour:
        # Subsample contour indices so the plot stays readable.
        idxs = np.linspace(0, cfg.n_contour_points - 1, min(max_strands, cfg.n_contour_points))
        idxs = np.unique(idxs.astype(int))
        for k, i in enumerate(idxs):
            xs = valid[f'contour_{i:03d}_x'].to_numpy()
            ys = valid[f'contour_{i:03d}_y'].to_numpy()
            color = cmap(k / max(len(idxs) - 1, 1))
            ax.plot(xs, ys, t, color=color, alpha=0.8, linewidth=1.0)

    if show_grid:
        # Use a single 'reference' grid point near the body centre to keep the figure tidy.
        r_mid, c_mid = cfg.grid_rows // 2, cfg.grid_cols // 2
        gx = valid[f'grid_r{r_mid:02d}_c{c_mid:02d}_x'].to_numpy()
        gy = valid[f'grid_r{r_mid:02d}_c{c_mid:02d}_y'].to_numpy()
        ax.plot(gx, gy, t, color='red', linewidth=2.0, label='central body point')
        ax.legend(loc='upper left', fontsize=9)

    # Centroid trajectory in red (thick) for orientation.
    ax.plot(valid['centroid_x'], valid['centroid_y'], t,
            color='black', linewidth=2.5, alpha=0.6, label='centroid')
    ax.set_xlabel('x (px)')
    ax.set_ylabel('y (px)')
    ax.set_zlabel('time (s)')
    ax.invert_yaxis()
    ax.set_title(f'3D space-time view of body surface · {video_id}')
    ax.view_init(elev=18, azim=-65)
    plt.tight_layout()
    plt.show()

# Example use right after a video is processed:
# plot_surface_spacetime(df, S, video_id=video_entries[0].relative_path)

## 9. Run on all videos

This is the equivalent of the YOLO notebook's final loop. The dispatcher walks every
`VideoEntry` discovered in section 4, applies any matching manual override from
`manual_targets.csv` automatically, and writes per-video outputs into
`outputs/<subject_id>/`.

After the loop, two extra files are produced in `OUTPUT_DIR`:

- **`_summary.csv`** — per-video sanity report. Columns: `subject_id`, `subject_type`,
  `video_relpath`, `n_frames`, `n_detected`, `detection_rate`, `mean_area`,
  `area_rsd` (relative SD = std / mean), `target_mode`. Sort by `detection_rate` to spot
  the videos where automatic patient selection may need a manual override.
- **`_all_timeseries.xlsx`** — every per-video Excel concatenated into one TabICLv2-ready
  table, with `subject_id` / `subject_type` already set.

In [ ]:
%%time
all_dfs: List[pd.DataFrame] = []
summary_rows: List[dict]    = []

for entry in video_entries:
    print('=' * 70)
    print(f'Start  : {entry.relative_path}  '
          f'(type={entry.subject_type}, '
          f'target={"MANUAL" if entry.manual_bbox else "auto"})')
    df = process_video_sam3(
        entry, predictor, S, OUTPUT_DIR,
        write_overlay=True, write_mask=True, write_sidebyside=True,
    )
    all_dfs.append(df)

    n_frames     = len(df)
    n_detected   = int(df.patient_detected.sum())
    det_rate     = n_detected / max(n_frames, 1)
    detected_df  = df[df.patient_detected == 1]
    if len(detected_df) > 0:
        mean_area  = float(detected_df.area_px.mean())
        area_rsd   = float(detected_df.area_px.std() / max(mean_area, 1))
    else:
        mean_area, area_rsd = float('nan'), float('nan')

    summary_rows.append(dict(
        subject_id     = entry.subject_id,
        subject_type   = entry.subject_type,
        video_relpath  = entry.relative_path,
        n_frames       = n_frames,
        n_detected     = n_detected,
        detection_rate = round(det_rate, 4),
        mean_area      = mean_area,
        area_rsd       = round(area_rsd, 4),
        target_mode    = 'MANUAL' if entry.manual_bbox is not None else 'auto',
    ))

    plot_video_features(df, S, video_id=entry.relative_path)
    plot_surface_spacetime(df, S, video_id=entry.relative_path)
    print(f'End    : {entry.relative_path}')
    print('=' * 70)

# ----------------------------------------------------------------------
# _summary.csv  --  per-video sanity report
# ----------------------------------------------------------------------
summary = pd.DataFrame(summary_rows).sort_values(
    by=['detection_rate', 'area_rsd'], ascending=[True, False]
)
summary_path = OUTPUT_DIR / '_summary.csv'
summary.to_csv(summary_path, index=False)
print(f'Wrote {summary_path}')

# Flag suspicious videos so the user knows which ones may need a manual override.
low_det = summary[summary.detection_rate < 0.85]
high_var = summary[(summary.area_rsd > 0.30) & (summary.detection_rate >= 0.85)]
if len(low_det) > 0:
    print('\nVideos with LOW detection rate (consider manual_targets.csv):')
    print(low_det[['video_relpath', 'detection_rate']].to_string(index=False))
if len(high_var) > 0:
    print('\nVideos with high area variability (likely OK but worth a quicklook):')
    print(high_var[['video_relpath', 'area_rsd']].to_string(index=False))

# ----------------------------------------------------------------------
# _all_timeseries.xlsx  --  concatenated table for TabICLv2
# ----------------------------------------------------------------------
if all_dfs:
    big = pd.concat(all_dfs, ignore_index=True)
    all_path = OUTPUT_DIR / '_all_timeseries.xlsx'
    # Excel has a 1,048,576-row hard limit; switch to parquet for big datasets.
    if len(big) <= 1_000_000:
        big.to_excel(all_path, index=False)
        print(f'Wrote {all_path}  ({len(big):,} rows, {big.shape[1]} columns)')
    else:
        all_path = all_path.with_suffix('.parquet')
        big.to_parquet(all_path, index=False)
        print(f'Dataset too large for Excel; wrote {all_path} instead.')

## 10. (Optional) Full 3D body reconstruction with SAM 3D Body

**Released 19 Nov 2025**, [SAM 3D Body](https://github.com/facebookresearch/sam-3d-body)
(`facebook/sam-3d-body-dinov3`) reconstructs a full 3D human mesh from a single image,
with the option to use 2D keypoints **and a binary mask** as prompts. Since we already
have a high-quality SAM 3 mask of the patient, we can feed it directly and get a 3D mesh
that is locked onto the patient — *not* the neurologist.

This section is **optional** and **independent** of the main pipeline above. Running
SAM 3D Body on every frame of a long clinical video is impractical (the model is heavy
and reconstruction takes several seconds per frame on a consumer GPU). We therefore
sample **keyframes** (default: 1 frame per second) and build a short 3D animation.

### Installation

From an Anaconda Prompt **outside** Jupyter, in the same `sam3` environment:

```bat
git clone https://github.com/facebookresearch/sam-3d-body.git
cd sam-3d-body
pip install -r requirements.txt   :: see INSTALL.md if pyrender / xtcocotools fail
pip install -e .

huggingface-cli login              :: accept the model licence on Hugging Face first
hf download facebook/sam-3d-body-dinov3 --local-dir checkpoints/sam-3d-body-dinov3
```

On Windows, if `xtcocotools` fails to build, install Microsoft Visual C++ Build Tools or
use `pip install --no-build-isolation xtcocotools`. PyTorch3D is **not strictly required**
for the visualisation we use here (we render with matplotlib's 3D scatter / trisurf).

Once installed, the cell below auto-detects whether SAM 3D Body is available and silently
skips the section otherwise.

In [ ]:
# ----------------------------------------------------------------------
# SAM 3D Body has no setup.py / pyproject.toml: we add its repo directory
# to sys.path manually. Adjust SAM_3D_BODY_REPO if you cloned it elsewhere.
# We also bypass pyrender (which requires EGL/OSMesa, not available on
# Windows): the notebook renders 3D meshes with matplotlib, not pyrender,
# so we replace pyrender with a MagicMock that absorbs all attribute
# accesses without crashing the import chain in renderer.py.
# ----------------------------------------------------------------------
import sys
from pathlib import Path as _Path
from unittest.mock import MagicMock as _MagicMock

# 1) Bypass pyrender BEFORE any sam-3d-body import.
if 'pyrender' not in sys.modules:
    sys.modules['pyrender'] = _MagicMock()

# 2) Add the sam-3d-body repo to sys.path. Edit this if your local clone
#    is elsewhere; default assumes a sibling layout next to this notebook.
SAM_3D_BODY_REPO = _Path(r'C:\Users\<user>\Desktop\sam_3\sam-3d-body')
if SAM_3D_BODY_REPO.is_dir() and str(SAM_3D_BODY_REPO) not in sys.path:
    sys.path.insert(0, str(SAM_3D_BODY_REPO))

# 3) Try the import.
SAM3D_BODY_AVAILABLE = False
try:
    from notebook.utils import setup_sam_3d_body  # noqa: E402
    SAM3D_BODY_AVAILABLE = True
    print('SAM 3D Body is importable. Section 10 will run.')
except Exception as e:
    print('SAM 3D Body is NOT importable -> section 10 will be skipped.')
    print(f'  reason: {e}')
    print('  (this is fine; the rest of the notebook does not depend on it)')

### 10.1. Load SAM 3D Body (only if available)

We load the DINOv3-H+ checkpoint (best quality) once and reuse it across all keyframes.
If you are VRAM-limited, switch to `facebook/sam-3d-body-vith` which is smaller.

In [ ]:
estimator_3d = None
if SAM3D_BODY_AVAILABLE:
    estimator_3d = setup_sam_3d_body(hf_repo_id='facebook/sam-3d-body-dinov3')
    print('SAM 3D Body estimator ready.')
else:
    print('Skipping load; SAM 3D Body not installed.')

### 10.2. Reconstruct the patient's 3D mesh on a single frame

We use the SAM 3 mask we already have as a prompt for SAM 3D Body. The function below:

1. Grabs one frame from a video,
2. Runs SAM 3 to obtain the patient mask (ignoring any other persons as before),
3. Crops to the patient's bounding box (with a small padding) for SAM 3D Body,
4. Calls `estimator_3d.process_one_image()` to get a mesh,
5. Plots the mesh in 3D with matplotlib.

The returned `outputs` dictionary contains the MHR vertex coordinates, joint positions and
the predicted camera. We focus on the vertices for visualisation; in clinical-research
use you can also store the per-vertex coordinates to disk for downstream feature
extraction.

In [ ]:
def reconstruct_3d_at_frame(
    video_path: Path,
    frame_idx:  int,
    predictor:  SAM3SemanticPredictor,
    estimator,
    cfg:        Settings,
    pad:        float = 0.10,        # bbox padding fraction
) -> Optional[dict]:
    """Reconstruct one 3D body mesh for one frame of one video.

    Returns
    -------
    {'frame_idx', 'time_s', 'outputs', 'crop'} or None on failure.
    """
    if estimator is None:
        print('SAM 3D Body is not loaded.')
        return None

    cap = cv2.VideoCapture(str(video_path))
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ok, frame = cap.read()
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    cap.release()
    if not ok:
        return None

    masks, boxes = sam3_segment_persons(frame, predictor, cfg.text_prompt)
    mask, box    = select_patient_mask(masks, boxes, None, cfg)
    if mask is None:
        return None

    H, W = frame.shape[:2]
    x1, y1, x2, y2 = box
    bw, bh = x2 - x1, y2 - y1
    x1 = max(0, int(x1 - pad * bw))
    y1 = max(0, int(y1 - pad * bh))
    x2 = min(W, int(x2 + pad * bw))
    y2 = min(H, int(y2 + pad * bh))
    crop = frame[y1:y2, x1:x2].copy()
    crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)

    # SAM 3D Body's process_one_image expects RGB. The mask prompt is optional but greatly
    # improves robustness when there are partial occlusions.
    try:
        outputs = estimator.process_one_image(crop_rgb)
    except Exception as e:
        print(f'  SAM 3D Body failed on frame {frame_idx}: {e}')
        return None

    return {
        'frame_idx': frame_idx,
        'time_s':    frame_idx / fps,
        'outputs':   outputs,
        'crop':      crop_rgb,
        'crop_origin': (x1, y1),
    }

In [ ]:
def _extract_mesh(outputs):
    """Best-effort extraction of (vertices, faces) from a SAM 3D Body output dict.

    The exact keys depend on the SAM 3D Body release. We try the most common ones and
    fall back to a simple search through the dict."""
    if outputs is None:
        return None, None

    # Most common case
    for vk in ('vertices', 'verts', 'mesh_vertices', 'pred_vertices'):
        if vk in outputs:
            v = outputs[vk]
            v = v[0] if hasattr(v, 'shape') and len(v.shape) == 3 else v
            if hasattr(v, 'detach'):
                v = v.detach().cpu().numpy()
            return np.asarray(v), outputs.get('faces', None)

    # Fallback: take the first ndarray-like value of shape (N, 3)
    for k, val in outputs.items():
        try:
            arr = val.detach().cpu().numpy() if hasattr(val, 'detach') else np.asarray(val)
            if arr.ndim == 2 and arr.shape[1] == 3 and arr.shape[0] > 100:
                return arr, outputs.get('faces', None)
            if arr.ndim == 3 and arr.shape[2] == 3 and arr.shape[1] > 100:
                return arr[0], outputs.get('faces', None)
        except Exception:
            continue
    return None, None

def plot_3d_mesh(result: dict, title: str = ''):
    """Display the reconstructed 3D mesh next to the input crop."""
    if result is None:
        print('No result to plot.')
        return

    verts, faces = _extract_mesh(result['outputs'])
    if verts is None:
        print('Could not find a vertex array in the SAM 3D Body output. Got keys:',
              list(result['outputs'].keys()))
        return

    fig = plt.figure(figsize=(13, 6))
    ax_img = fig.add_subplot(1, 2, 1)
    ax_img.imshow(result['crop'])
    ax_img.set_title(f't = {result["time_s"]:.2f}s — input crop'); ax_img.axis('off')

    ax3d = fig.add_subplot(1, 2, 2, projection='3d')
    if faces is not None:
        try:
            ax3d.plot_trisurf(
                verts[:, 0], verts[:, 1], verts[:, 2],
                triangles=np.asarray(faces),
                color='lightcoral', alpha=0.85, edgecolor='none', linewidth=0,
            )
        except Exception:
            ax3d.scatter(verts[:, 0], verts[:, 1], verts[:, 2], s=1, c='steelblue')
    else:
        ax3d.scatter(verts[:, 0], verts[:, 1], verts[:, 2], s=1, c='steelblue')

    # SMPL-like meshes have y down (image convention) — flip for a nice viewer angle.
    ax3d.set_xlabel('x'); ax3d.set_ylabel('y'); ax3d.set_zlabel('z')
    ax3d.set_box_aspect((1, 1, 1))
    ax3d.view_init(elev=10, azim=-80)
    ax3d.set_title(f'SAM 3D Body mesh ({len(verts)} vertices)')
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

### 10.3. Sanity check on one frame

In [ ]:
if SAM3D_BODY_AVAILABLE and len(video_entries) > 0:
    res = reconstruct_3d_at_frame(
        video_entries[0].path, frame_idx=0,
        predictor=predictor, estimator=estimator_3d, cfg=S,
    )
    plot_3d_mesh(res, title=f'SAM 3D Body  -  {video_entries[0].relative_path}')
else:
    print('Skipping (SAM 3D Body not available or no videos).')

### 10.4. Build a 3D animation across the whole video

Reconstruct the patient's 3D mesh at one keyframe per second, then export an MP4 where
the camera rotates around the mesh while the body deforms (e.g. shows tremor amplitude).

On a consumer RTX GPU (≥ 8 GB VRAM), expect roughly 2–5 s per keyframe. For a 30-second
video at 1 keyframe/s this is ~1–2 minutes per video.

In [ ]:
import matplotlib.animation as manim
from matplotlib.colors import LightSource

def build_3d_mesh_animation(
    entry,                          # VideoEntry, NOT a Path: gives us subject_id
    predictor,                      # SAM3SemanticPredictor
    estimator,                      # SAM 3D Body estimator
    cfg,                            # Settings
    output_dir,                     # Path
    keyframe_every_seconds: float = 1.0,
    fps_out:                int   = 12,
    rotate_camera:          bool  = True,
    save_vertices:          bool  = True,
):
    """Reconstruct the patient's 3D body mesh on keyframes and save an MP4
    animation alongside an optional .npz of the per-keyframe vertices.

    Outputs (in output_dir/<subject_id>/):
      <stem>_sam3body_3d.mp4          rotating 3D body animation, side-by-side
                                       with the input crop
      <stem>_sam3body_3d.npz          saved keyframe vertices and metadata
                                       (only if save_vertices=True)
    """
    if estimator is None:
        print('SAM 3D Body is not loaded.'); return None

    video_path = entry.path
    cap = cv2.VideoCapture(str(video_path))
    fps_in = cap.get(cv2.CAP_PROP_FPS) or 30.0
    nframes = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    step = max(1, int(round(fps_in * keyframe_every_seconds)))
    keyframes = list(range(0, nframes, step))
    print(f'  reconstructing on {len(keyframes)} keyframes (every '
          f'{keyframe_every_seconds:.1f}s, total {nframes/fps_in:.1f}s of video)...')

    # ------------------------------------------------------------------
    # 1) Run SAM 3D Body on every keyframe.
    # ------------------------------------------------------------------
    t0 = time.time()
    results = []
    for k, fi in enumerate(keyframes, 1):
        r = reconstruct_3d_at_frame(video_path, fi, predictor, estimator, cfg)
        if r is not None:
            results.append(r)
        if k % 5 == 0 or k == len(keyframes):
            elapsed = time.time() - t0
            rate = k / max(elapsed, 1e-6)
            eta  = (len(keyframes) - k) / max(rate, 1e-6)
            print(f'    {k:3d}/{len(keyframes)}  rate={rate:.2f} kf/s  ETA={eta:.0f}s')

    if not results:
        print('  no keyframe could be reconstructed.'); return None

    # ------------------------------------------------------------------
    # 2) Pre-compute axes range so the 3D view does not jiggle frame to frame.
    # ------------------------------------------------------------------
    all_verts = []
    for r in results:
        v, _ = _extract_mesh(r['outputs'])
        if v is not None:
            all_verts.append(v)
    if not all_verts:
        print('  no extractable mesh.'); return None
    stacked = np.concatenate(all_verts, axis=0)
    span = (stacked.max(axis=0) - stacked.min(axis=0)).max()
    centre = (stacked.max(axis=0) + stacked.min(axis=0)) / 2
    half = span / 2 * 1.1
    xlim = (centre[0] - half, centre[0] + half)
    ylim = (centre[1] - half, centre[1] + half)
    zlim = (centre[2] - half, centre[2] + half)

    # ------------------------------------------------------------------
    # 3) Build the figure: input crop on the left, 3D mesh on the right.
    # ------------------------------------------------------------------
    fig = plt.figure(figsize=(12, 6), facecolor='white')
    ax_img = fig.add_subplot(1, 2, 1)
    ax3d   = fig.add_subplot(1, 2, 2, projection='3d')
    ax3d.set_facecolor('white')

    light = LightSource(azdeg=315, altdeg=45)

    def update(idx):
        ax_img.clear(); ax3d.clear()
        r = results[idx]

        ax_img.imshow(r['crop']); ax_img.axis('off')
        ax_img.set_title(f't = {r["time_s"]:.2f}s   (frame {r["frame_idx"]})',
                         fontsize=11)

        verts, faces = _extract_mesh(r['outputs'])
        if verts is None:
            return

        try:
            if faces is not None:
                # Shaded surface for a more "3D-looking" render.
                ax3d.plot_trisurf(
                    verts[:, 0], verts[:, 1], verts[:, 2],
                    triangles=np.asarray(faces),
                    color='lightcoral', alpha=0.95,
                    edgecolor='none', linewidth=0,
                    shade=True, lightsource=light,
                )
            else:
                ax3d.scatter(verts[:, 0], verts[:, 1], verts[:, 2],
                             s=2, c='steelblue', alpha=0.5)
        except Exception:
            ax3d.scatter(verts[:, 0], verts[:, 1], verts[:, 2],
                         s=2, c='steelblue', alpha=0.5)

        ax3d.set_xlim(xlim); ax3d.set_ylim(ylim); ax3d.set_zlim(zlim)
        ax3d.set_box_aspect((1, 1, 1))
        if rotate_camera:
            azim = -80 + 90 * idx / max(len(results) - 1, 1)
        else:
            azim = -80
        ax3d.view_init(elev=10, azim=azim)

        # Clean axes
        ax3d.set_xticklabels([]); ax3d.set_yticklabels([]); ax3d.set_zticklabels([])
        ax3d.grid(False)
        ax3d.xaxis.pane.set_alpha(0); ax3d.yaxis.pane.set_alpha(0); ax3d.zaxis.pane.set_alpha(0)
        ax3d.set_title(f'3D body mesh  ·  {entry.subject_id}  ({entry.subject_type})',
                       fontsize=11)

        fig.suptitle(f'{entry.relative_path}', fontsize=10, y=0.99)

    anim = manim.FuncAnimation(fig, update, frames=len(results), blit=False)

    # ------------------------------------------------------------------
    # 4) Save the animation in outputs/<subject_id>/.
    # ------------------------------------------------------------------
    subject_dir = output_dir / entry.subject_id
    subject_dir.mkdir(parents=True, exist_ok=True)

    stem     = video_path.stem
    out_mp4  = subject_dir / f'{stem}_sam3body_3d.mp4'
    out_npz  = subject_dir / f'{stem}_sam3body_3d.npz'

    saved_path = None
    try:
        anim.save(str(out_mp4),
                  writer=manim.FFMpegWriter(fps=fps_out, bitrate=2400),
                  dpi=110)
        saved_path = out_mp4
        print(f'  -> 3D animation saved: {out_mp4}')
    except Exception as e:
        out_gif = out_mp4.with_suffix('.gif')
        anim.save(str(out_gif), writer=manim.PillowWriter(fps=fps_out), dpi=110)
        saved_path = out_gif
        print(f'  ffmpeg unavailable ({e}); saved GIF instead: {out_gif}')
    plt.close(fig)

    # ------------------------------------------------------------------
    # 5) Save per-keyframe vertices for downstream analysis.
    # ------------------------------------------------------------------
    if save_vertices and all_verts:
        try:
            np.savez_compressed(
                out_npz,
                vertices       = np.stack(all_verts, axis=0),  # (K, V, 3)
                frame_indices  = np.array([r['frame_idx'] for r in results]),
                times_s        = np.array([r['time_s']    for r in results]),
                subject_id     = str(entry.subject_id),
                subject_type   = str(entry.subject_type),
                video_relpath  = str(entry.relative_path),
            )
            print(f'  -> vertices saved:     {out_npz}  '
                  f'({len(all_verts)} keyframes x {all_verts[0].shape[0]} vertices)')
        except Exception as e:
            print(f'  -> WARNING could not save vertices npz: {e}')

    return saved_path


### 10.5. Demo run — 3D body animation on a small selection

By default this notebook does **not** run SAM 3D Body on every video — that
would take many extra hours for a typical clinical batch. Instead, this cell
picks **2 patients and 1 control** and produces a fully rendered 3D body
animation for the *first* video of each.

Each run produces, in `outputs/<subject_id>/`:

- `<stem>_sam3body_3d.mp4` — side-by-side animation: input crop on the left,
  rotating 3D body mesh on the right (one keyframe per second by default).
- `<stem>_sam3body_3d.npz` — per-keyframe vertex coordinates and timestamps,
  for downstream analysis without re-running SAM 3D Body.

Edit the lists `PATIENTS_TO_RUN` and `CONTROLS_TO_RUN` below if you want
specific subjects rather than the first three found.

**Expected runtime on a single RTX 5080**: ~1–3 minutes per video, depending
on duration and `keyframe_every_seconds`.

In [ ]:
# ----------------------------------------------------------------------
# Demo: 2 patients + 1 control. Edit the lists below to pick specific
# subjects, or leave them as None to take the first that comes up.
# ----------------------------------------------------------------------
PATIENTS_TO_RUN: list = []      # e.g. ['P1_RPA', 'P12_RPA']  -- empty = pick first 2
CONTROLS_TO_RUN: list = []      # e.g. ['C5_RPA']              -- empty = pick first 1

N_PATIENTS = 2
N_CONTROLS = 1

# Pick keyframe density. 1.0 = one mesh per second; lower for finer motion,
# higher to save time. Beware: SAM 3D Body is heavy.
KEYFRAME_EVERY_SECONDS = 1.0
FPS_OUT                = 12     # frames per second of the output animation

if not SAM3D_BODY_AVAILABLE or estimator_3d is None:
    print('SAM 3D Body is not available -- skipping the 3D demo.')
else:
    # ------------------------------------------------------------------
    # Build the list of subjects to process.
    # ------------------------------------------------------------------
    by_type = {'patient': [], 'control': []}
    seen    = set()
    for e in video_entries:
        if e.subject_id not in seen:
            by_type[e.subject_type].append(e.subject_id)
            seen.add(e.subject_id)

    if not PATIENTS_TO_RUN:
        PATIENTS_TO_RUN = by_type['patient'][:N_PATIENTS]
    if not CONTROLS_TO_RUN:
        CONTROLS_TO_RUN = by_type['control'][:N_CONTROLS]

    targets = set(PATIENTS_TO_RUN) | set(CONTROLS_TO_RUN)

    # For each target subject, take the FIRST video alphabetically.
    selected = []
    for sid in PATIENTS_TO_RUN + CONTROLS_TO_RUN:
        videos_of_sid = sorted(
            (e for e in video_entries if e.subject_id == sid),
            key=lambda e: e.path.name,
        )
        if videos_of_sid:
            selected.append(videos_of_sid[0])
        else:
            print(f'WARNING: subject {sid} has no videos -- skipping')

    print(f'\nWill run 3D body reconstruction on {len(selected)} videos:')
    for e in selected:
        print(f'  - {e.relative_path}  ({e.subject_type})')
    print()

    # ------------------------------------------------------------------
    # Run the 3D batch.
    # ------------------------------------------------------------------
    t_total_start = time.time()
    for i, entry in enumerate(selected, 1):
        print('=' * 70)
        print(f'[{i}/{len(selected)}] 3D body animation for {entry.relative_path}')
        try:
            build_3d_mesh_animation(
                entry, predictor, estimator_3d, S, OUTPUT_DIR,
                keyframe_every_seconds = KEYFRAME_EVERY_SECONDS,
                fps_out                = FPS_OUT,
                rotate_camera          = True,
                save_vertices          = True,
            )
        except Exception as e:
            import traceback
            print(f'  FAILED on {entry.relative_path}: {e}')
            traceback.print_exc()
        print('=' * 70)

    print(f'\nTotal 3D demo time: {(time.time() - t_total_start)/60:.1f} minutes')


## 11. Next step — bridge to TabICLv2

The Excel files produced above already conform to the *one-row-per-frame, fixed-schema*
contract that the TabICLv2 prediction pipeline expects. To plug them in:

1. **Concatenate** all `<video_id>_sam3_timeseries.xlsx` files (add a `video_id` column).
2. **Window the signals** with the same 10-second / 300-frame sliding window used in
   Cif & Vasques (2026) (300-frame length, 150-frame stride at 30 fps).
3. **For each window and each signal column** (centroid, area, contour points, grid
   points, …), compute the same 19 kinematic descriptors as in the YOLO pipeline:
   distributional (mean, std, min, max, range, IQR, …), temporal (slope, energy, mean
   absolute acceleration, derivative zero-crossings), spectral (dominant peak frequency
   and amplitude) and complexity (Higuchi fractal dimension, permutation entropy,
   histogram-based entropy).
4. **Normalise** with median-and-IQR scaling at the signal level.
5. **Feed** the resulting tabular matrix to TabICLv2, with the same one-vs-rest setup for
   the eight target phenomenologies (dystonia, tremor, myoclonus, chorea, athetosis,
   ballismus, stereotypies, tics).

The number of signals per frame is now much larger than with YOLOv8 Pose:

| Source              | YOLOv8 Pose | SAM 3 (this notebook)                                |
|---------------------|-------------|------------------------------------------------------|
| Keypoints / signals | 17 (x, y) → 34 | 14 geom. + 64 contour (×2) + 96 grid (×2) → **334**  |

If the optional SAM 3D Body section is also run, you additionally get **~5–7 K mesh
vertices in 3D per keyframe**, which can be summarised into per-region descriptors
(torso, arms, legs, head, hands) and concatenated to the table.

This much wider tabular input is the whole point of switching from sparse pose to dense
surface description: it exposes regional movement information (chest, abdomen, limbs,
head outline) that is invisible to a 17-keypoint skeleton, which is a likely benefit for
phenomenologies like *dystonia* and *chorea* that involve continuous postural deformation
rather than discrete joint motion.

## Appendix — Troubleshooting

**`OutOfMemoryError: CUDA out of memory`.** Lower `S.imgsz` (768 → 640 → 512). If still
OOM, fall back to SAM 2 by replacing `SAM3SemanticPredictor` with the SAM 2 visual
predictor and segmenting from a manual bounding box of the patient on frame 0.

**`TypeError: 'SimpleTokenizer' object is not callable`.** The wrong `clip` package is
installed. In your activated environment:

```bat
pip uninstall clip -y
pip install git+https://github.com/ultralytics/CLIP.git
```

**SAM 3 picks up the neurologist instead of the patient.** Set `S.manual_bbox` to the
patient's bounding box on frame 0 (e.g. `(120, 80, 540, 720)`) before running
`process_video_sam3()` — patient identity is then propagated through the whole video by
IoU continuity.

**Inference is too slow.** Increase `S.frame_stride` to 2 or 3 (process every 2nd / 3rd
frame). The 10-second-window feature extractor in step 11 above tolerates this without
loss of clinical information.

**`ffmpeg` not found when saving the 3D animation.** On Windows, install via
`conda install -c conda-forge ffmpeg`. The notebook automatically falls back to a GIF if
ffmpeg is unavailable.